# Colab Fine-Tuning Launcher

This notebook wires up the existing `scripts/finetune.py` entrypoint so it can be run from Google Colab with Drive-backed paths.

Important: the training dataset must be disjoint from the videos used at inference time.

If the repository is not already available in Colab, clone it under `/content/computer-vision-project` before running the cells below.

## 1. Set Up Imports and Configuration

In [ ]:
!pip install -q ultralytics roboflow

In [ ]:
from __future__ import annotations

import os
import shlex
from dataclasses import dataclass
from pathlib import Path

# Colab-only mount. If you run this notebook elsewhere, skip this cell.
try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount("/content/drive")

REPO_ROOT = Path("/content/computer-vision-project")
DRIVE_ROOT = Path("/content/drive/MyDrive/computer-vision-project")
DATA_YAML = DRIVE_ROOT / "data" / "roboflow_export" / "data.yaml"
MODEL_PATH = REPO_ROOT / "models" / "yolo11m.pt"
EPOCHS = 50
IMGSZ = 1280
BATCH = 8
DEVICE = 0
RUN_NAME = "yolo11m_finetune"
OUT_PATH = REPO_ROOT / "models" / "yolo11m_finetuned.pt"

## 2. Define Core Data Structures

In [ ]:
@dataclass
class FineTuneConfig:
    repo_root: Path = REPO_ROOT
    drive_root: Path = DRIVE_ROOT
    data_yaml: Path = DATA_YAML
    model_path: Path = MODEL_PATH
    epochs: int = EPOCHS
    imgsz: int = IMGSZ
    batch: int = BATCH
    device: int | str = DEVICE
    run_name: str = RUN_NAME
    out_path: Path = OUT_PATH

    def to_command(self) -> list[str]:
        return [
            "python",
            "scripts/finetune.py",
            "--data",
            str(self.data_yaml),
            "--model",
            str(self.model_path),
            "--epochs",
            str(self.epochs),
            "--imgsz",
            str(self.imgsz),
            "--batch",
            str(self.batch),
            "--device",
            str(self.device),
            "--name",
            self.run_name,
            "--out",
            str(self.out_path),
        ]
config = FineTuneConfig()

## 3. Implement the First Core Function

In [ ]:
def mount_drive() -> None:
    if drive is None:
        raise RuntimeError("This notebook is intended to run in Google Colab.")
    drive.mount("/content/drive")


def ensure_paths(cfg: FineTuneConfig) -> None:
    if not cfg.data_yaml.exists():
        raise FileNotFoundError(f"Missing data.yaml: {cfg.data_yaml}")
    if not cfg.model_path.exists():
        raise FileNotFoundError(f"Missing starting weights: {cfg.model_path}")
    cfg.out_path.parent.mkdir(parents=True, exist_ok=True)


def launch_finetune(cfg: FineTuneConfig) -> list[str]:
    ensure_paths(cfg)
    return cfg.to_command()

## 4. Add Input Validation and Error Handling

In [ ]:
try:
    mount_drive()
except RuntimeError as exc:
    print(exc)

print(f"Repository root: {config.repo_root}")
print(f"Drive root: {config.drive_root}")
print(f"Data YAML: {config.data_yaml}")
print(f"Model path: {config.model_path}")
print(f"Output path: {config.out_path}")

if config.repo_root.exists():
    print("Repository checkout found.")
else:
    print("Repository checkout not found yet. Clone the repo into /content/computer-vision-project before launching training.")

if config.data_yaml.exists():
    print("Dataset configuration found.")

## 5. Run an Example and Verify Output

In [ ]:
command = launch_finetune(config)
print("Training command:")
print(" ".join(shlex.quote(part) for part in command))

# In Colab, run the next cell after confirming the paths above.
# Example execution command:
# !python scripts/finetune.py --data ...

In [ ]:
import subprocess

result = subprocess.run(command, cwd=str(config.repo_root), check=True)
print(f"Finetuning finished with exit code {result.returncode}")
print(f"Expected trained model at: {config.out_path}")
if config.out_path.exists():
    print("Trained model artifact found.")
else:
    print("Trained model artifact not found yet. Check the training logs and output path.")

## 6. Create Basic Tests

In [ ]:
test_command = launch_finetune(config)
assert config.repo_root.name == "computer-vision-project"
assert config.data_yaml.suffix == ".yaml"
assert config.out_path.name.endswith(".pt")
assert test_command[0] == "python"
assert "scripts/finetune.py" in test_command
print("Basic notebook tests passed.")